In [ ]:
!pip install numpy sentence-transformers ripser tqdm requests pandas statsmodels stargazer

In [ ]:
!pip install transformers==4.51.3 torch accelerate

In [ ]:
!pip install vllm

# Сбор графов

In [ ]:
taxonomies = {
    "math": {
        "types": ["Given", "Lemma", "Inference", "Substitution", "Simplification", "Result", "Assumption", "Contradiction"],
        "description": "Математическое рассуждение: дано → лемма → вывод → подстановка → упрощение → результат"
    },
    "law": {
        "types": ["Fact", "Norm", "Precedent", "Inference", "Assumption", "Conclusion", "Contradiction", "ExternalReference"],
        "description": "Юридическое рассуждение: факт → норма права → прецедент → вывод → заключение"
    },
    "general": {
        "types": ["Premise", "Inference", "Assumption", "Evidence", "Conclusion", "Contradiction", "ExternalKnowledge"],
        "description": "Общее логическое рассуждение"
    }
}

def build_taxonomy_prompt(domain="general"):
    tax = taxonomies[domain]
    types_str = ", ".join(tax["types"])
    return f"""Типы узлов: {types_str}
      Описание: {tax['description']}

      Каждый тип означает:
      - Given/Fact: исходные данные, не требующие доказательства
      - Lemma/Norm: вспомогательное утверждение/норма права
      - Inference: логический переход от одного к другому
      - Assumption: допущение (может быть явным или неявным)
      - Result/Conclusion: финальный ответ
      - Contradiction: обнаруженное противоречие
      - ExternalReference/ExternalKnowledge: ссылка на внешний источник
      """

In [ ]:
import os

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset
import torch
import gc
import json
import time
from pathlib import Path
from typing import Any, List
import re
from tqdm import tqdm
import pandas as pd


def ensure_dir(path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

def setup_vllm_model(model_name: str, gpu_memory_utilization: float = 0.85,
                     max_model_len=5000):

    print(f"Loading {model_name} with vLLM...")

    llm = LLM(
        model=model_name,
        tensor_parallel_size=1,
        trust_remote_code=True,
        max_model_len=max_model_len,
        dtype="float16",
        quantization="awq_marlin",
        gpu_memory_utilization=gpu_memory_utilization,
        disable_log_stats=True,
        enforce_eager=False
    )

    tokenizer = llm.get_tokenizer()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Model loaded successfully")
    return llm, tokenizer

def generate_batch_vllm(
    llm: LLM,
    prompts: List[str],
    max_tokens: int = 3800,
    temperature: float = 0.0
) -> List[str]:
    sampling_params = SamplingParams(
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=1.0,
        top_k=-1,
        stop=None,
        repetition_penalty=1.0
    )


    outputs = llm.generate(prompts, sampling_params)
    responses = [output.outputs[0].text for output in outputs]

    return responses

def build_graph_prompt(question: str, cot: str, text_id, taxonomies, domain: str = "general") -> str:
        tax = taxonomies.get(domain, taxonomies["general"])

        return f"""Ты - Анализатор структуры рассуждений (Reasoning Graph Constructor).

          Задача: из данного Chain‑of‑Thought (CoT) извлечь граф логических элементов, который отражает
          НЕЛИНЕЙНУЮ структуру рассуждений: ветвления, перекрёстные ссылки, контраргументы, уточнения, условные выводы.

          Вопрос: {question}

          Рассуждение (CoT):
          {cot}

          Требования к графу:
          - Узлы (nodes) - очень обобщенные смысловые фрагменты, но не просто формулы или словосочетания,
              а краткое и емкое описание словами того, что происходит внутри шага
          - Рёбра (edges) могут быть любых типов из списка ниже.
          - Граф может быть неацикличным (допустимы обратные связи, повторные использования фактов).
          - Допускаются узлы без связей (если фрагмент изолирован).
          - Добавляй мета-информацию в каждый узел (тип роли: claim, evidence, assumption, rebuttal, synthesis).

          Типы рёбер (relation):
          - SUPPORTS - подтверждает/аргументирует
          - ATTACKS - противоречит / подрывает
          - EQUIV - эквивалентен / переформулирует
          - LEADS_TO - логическое следствие
          - CHALLENGES - ставит под сомнение, но не полностью опровергает
          - REFINES - уточняет / конкретизирует
          - ABSTRACTION - обобщает / абстрагирует
          - IF_TRUE_THEN - условная связь (если A истинно, то следует B)
          - ASSUMES - узел B неявно полагается на A как на предпосылку
          - CONTRADICTS - явное логическое противоречие
          - SYNTHESIS - синтезирует два или более узла в новый вывод
          - PARALLEL - независимые ветки одного уровня (метасвязь)

          Формат вывода - ТОЛЬКО JSON (без пояснений). Отдай ответ в таком виде, чтобы его можно было сразу преобразовать в json в Python.

          Структура:
          {{
            "nodes": [
              {{ "id": "n1", "text": "...", "role": "claim"}},
              {{ "id": "n2", "text": "...", "role": "evidence"}}
            ],
            "edges": [
              {{ "source": "n1", "target": "n2", "relation": "LEADS_TO" }}
            ],
            "metadata": {{
              "has_cycles": false,
              "is_linear": true,
              "total_reasoning_paths": 2
            }}
          }}

          Правила:
          1. Сохраняй порядок, но не принуждай к линейности.
          2. Если один и тот же факт используется в разных частях рассуждения - создавай отдельные узлы ИЛИ одну вершину
          с несколькими входящими/исходящими рёбрами (решай по смыслу).
          3. Не добавляй узлов, которых нет в CoT (не домысливай).
          4. Если тип связи однозначно не определить - ставь "SUPPORTS" или "REFINES"."""

def prepare_prompts(rows: List[dict], tokenizer, prompt: str) -> List[str]:
    prompts = []
    for ex in rows:
        statement = ex.get("statement", "")
        messages=[
                {"role": "system", "content": "Ты аналитик, раскладывающий рассуждения на графы."},
                {"role": "user", "content": prompt}
            ],
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        prompts.append(prompt)
    return prompts

def main():
    model_name = "Qwen/Qwen2.5-32B-Instruct-AWQ"

    print("=" * 60)

    all_prompts = []

    traces_path = "/content/all_traces_math_qwen32b_vllm_final.jsonl"

    out = "/content/llm_graphs_math_qwen2.5_32b_final.jsonl"
    batch_size = 16
    max_new_tokens = 3800

    all_traces = []

    with open(traces_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                trace =  json.loads(line)
                all_traces.append(trace)

    traces_df = pd.DataFrame(all_traces)

    print("\n" + "=" * 60)
    print("Setting up main model with vLLM...")
    print("=" * 60)
    model, tokenizer = setup_vllm_model(model_name, gpu_memory_utilization=0.7)
    print("\n" + "=" * 60)
    print("Preparing prompts...")

    all_prompts = []
    for id, row in traces_df.iterrows():
      prompt = build_graph_prompt(row['statement'], row['trace'], row['id'], taxonomies, domain = "math")
      all_prompts.append(prompt)

    rows = [row for _, row in traces_df.iterrows()]

    print("\n" + "=" * 60)
    print(f"Generating responses in batches of {batch_size}...")
    print("=" * 60)

    all_responses = []

    start_total = time.time()

    for batch_start in tqdm(range(len(rows), batch_size), desc="Generating batches"):
        batch_end = min(batch_start + batch_size, len(all_prompts))
        batch_prompts = all_prompts[batch_start:batch_end]
        batch_rows = rows[batch_start:batch_end]

        batch_start_time = time.time()
        try:
            responses = generate_batch_vllm(
                model,
                batch_prompts,
                max_tokens=max_new_tokens,
                temperature=0.0
            )
            gen_time = time.time() - batch_start_time

            for idx, (tr, response) in enumerate(zip(batch_rows, responses)):

                print(f"\nProblem {batch_start + idx + 1}:")
                print(f"Predicted graph: {response}")

                trace = {
                    "id": tr.get("id"),
                    "statement": tr.get("statement"),
                    "model": model_name,
                    "trace": tr.get("trace"),
                    "graph": response,
                    "reference_solution": tr.get("solution"),
                    "is_correct": tr.get("is_correct"),
                    "timestamp": int(time.time()),
                    "processing_time": gen_time / len(batch_prompts)
                }
                all_traces.append(trace)
                all_responses.append(response)

                with open(out, "a", encoding="utf-8") as out_f:
                    out_f.write(json.dumps(trace, ensure_ascii=False) + "\n")
                    out_f.flush()

        except Exception as e:
            print(f"Batch generation error: {e}")
            for ex in batch_rows:
                trace = {
                    "id": ex.get("id"),
                    "statement": ex.get("statement"),
                    "model": model_name,
                    "trace": f"ERROR: {str(e)}",
                    "pred_answer": 'invalid',
                    "final_answer": ex.get("final_answer"),
                    "reference_solution": ex.get("solution"),
                    "is_correct": 'No',
                    "timestamp": int(time.time()),
                    "processing_time": 0
                }
                all_traces.append(trace)
                all_responses.append(('invalid', ex.get("final_answer")))

        if batch_start % (batch_size * 4) == 0:
            print(f"\nGPU memory after batch {batch_start//batch_size + 1}:")
            for i in range(torch.cuda.device_count()):
                allocated = torch.cuda.memory_allocated(i) / 1e9
                print(f"  GPU {i}: {allocated:.2f}GB allocated")


    total_gen_time = time.time() - start_total


    print("\n" + "=" * 60)
    print("FINAL STATISTICS")
    print("=" * 60)

    print(f"Total problems: {len(all_traces)}")

    print(f"\nGeneration speed: {len(all_traces)/total_gen_time:.2f} samples/sec")
    print(f"\nResults saved to: {out}")

if __name__ == "__main__":
    main()

In [ ]:
from google.colab import files

files.download('/content/llm_graphs_math_qwen2.5_32b_final.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Парсинг графа

In [ ]:
import pandas as pd
import json

llm_graphs_df = pd.read_csv('/content/llm_graphs_math_qwen2.5_32b_final.csv')

In [ ]:
def extract_json(text):
    brace_count = 0
    start = None

    for i, char in enumerate(text):
        if char == '{':
            if brace_count == 0:
                start = i
            brace_count += 1
        elif char == '}':
            brace_count -= 1
            if brace_count == 0 and start is not None:
                json_str = text[start:i+1]
                try:
                    return json.loads(json_str)
                except json.JSONDecodeError:
                    continue
    return 'invalid'

In [ ]:
def is_path_graph(G):

    G_und = G.to_undirected()
    if not nx.is_connected(G_und):
        return False

    if nx.number_of_selfloops(G) > 0:
        return False

    odd_degree_nodes = [v for v, degree in G.degree() if degree != 2]
    if len(odd_degree_nodes) == 0 and G.number_of_nodes() == 1:
        return True

    degrees = [d for n, d in G.degree()]
    if set(degrees) not in [{1}, {2}, {1, 2}]:
        return False

    if degrees.count(1) == 2:
        return True
    else:
        return False

In [ ]:
llm_graphs_df['graph'] = llm_graphs_df['graph'].apply(lambda text: extract_json(text))

In [ ]:
llm_graphs_df[llm_graphs_df['graph'] == 'invalid']

,Unnamed: 0,id,statement,model,trace,graph,reference_solution,is_correct,timestamp,processing_time
8,8,9,The sequence of integers in the row of squares...,Qwen/Qwen2.5-32B-Instruct-AWQ,"To solve the problem, we need to determine th...",invalid,NaN,No,1778932026,1.906247
9,9,10,Tim wants to invest some money in a bank which...,Qwen/Qwen2.5-32B-Instruct-AWQ,"To solve this problem, we need to use the for...",invalid,NaN,No,1778932026,1.906247
11,11,12,Consider the given functions: $$\begin{array}{...,Qwen/Qwen2.5-32B-Instruct-AWQ,To solve for the value of \( k \) given the f...,invalid,NaN,Yes,1778932026,1.906247
81,81,82,Compute $\frac{x^8+12x^4+36}{x^4+6}$ when $x=5$.,Qwen/Qwen2.5-32B-Instruct-AWQ,"To solve the problem, we will follow these st...",invalid,NaN,Yes,1778932141,1.457989
111,111,112,"If $x$, $y$, and $z$ are positive numbers sati...",Qwen/Qwen2.5-32B-Instruct-AWQ,To solve for \(xyz\) given the equations:\n\[...,invalid,NaN,Yes,1778932164,1.396631
...,...,...,...,...,...,...,...,...,...,...
12388,5972,5282,"Consider a string of $n$ $7$'s, $7777\cdots77,...",Qwen/Qwen2.5-32B-Instruct-AWQ,\n\nStep 1: Understanding the problem\nWe are...,invalid,NaN,No,1778970599,2.353718
12390,5974,5284,Three clever monkeys divide a pile of bananas....,Qwen/Qwen2.5-32B-Instruct-AWQ,\n\nLet's denote the total number of bananas ...,invalid,NaN,No,1778970599,2.353718
12414,5998,5308,Define $n!!$ to be $n(n-2)(n-4)\cdots 3\cdot 1...,Qwen/Qwen2.5-32B-Instruct-AWQ,\n\nStep 1: Understanding the Double Factoria...,invalid,NaN,No,1778970633,2.153590
12415,5999,5309,Let $N$ be the number of ways to write $2010$ ...,Qwen/Qwen2.5-32B-Instruct-AWQ,"To solve the problem, we need to find the num...",invalid,NaN,No,1778970633,2.153590


In [ ]:
llm_graphs_df[llm_graphs_df['id'] == 735]['graph'].values[0]

{'nodes': [{'id': 'n1',
   'text': 'Substitute a = 2 into the product',
   'role': 'evidence'},
  {'id': 'n2',
   'text': 'Identify the terms in the product',
   'role': 'evidence'},
  {'id': 'n3',
   'text': 'Determine if any of the terms are zero',
   'role': 'evidence'},
  {'n4': 'n4', 'text': 'The product includes the term 0', 'role': 'evidence'},
  {'n5': 'n5',
   'text': 'Any number multiplied by zero is zero',
   'role': 'evidence'},
  {'n6': 'n6', 'text': 'The value of the product is 0', 'role': 'claim'}],
 'edges': [{'source': 'n1', 'target': 'n2', 'relation': 'LEADS_TO'},
  {'source': 'n2', 'target': 'n3', 'relation': 'LEADS_TO'},
  {'source': 'n3', 'target': 'n4', 'relation': 'SUPPORTS'},
  {'source': 'n4', 'target': 'n5', 'relation': 'SUPPORTS'},
  {'source': 'n5', 'target': 'n6', 'relation': 'LEADS_TO'}],
 'metadata': {'has_cycles': False,
  'is_linear': True,
  'total_reasoning_paths': 1}}

In [ ]:
llm_graphs_df.head(5)

,Unnamed: 0,id,statement,model,trace,graph,reference_solution,is_correct,timestamp,processing_time
0,0,1,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",Qwen/Qwen2.5-32B-Instruct-AWQ,To ensure the piecewise function \( f(x) \) i...,"{'nodes': [{'id': 'n1', 'text': 'Check continu...",NaN,Yes,1778932026,1.906247
1,1,2,A rectangular band formation is a formation wi...,Qwen/Qwen2.5-32B-Instruct-AWQ,"To solve the problem, we need to find the lar...","{'nodes': [{'id': 'n1', 'text': 'Initial forma...",NaN,Yes,1778932026,1.906247
2,2,3,What is the degree of the polynomial $(4 +5x^3...,Qwen/Qwen2.5-32B-Instruct-AWQ,To determine the degree of the polynomial \(4...,"{'nodes': [{'id': 'n1', 'text': 'Combine const...",NaN,Yes,1778932026,1.906247
3,3,4,Evaluate $\left\lceil3\left(6-\frac12\right)\r...,Qwen/Qwen2.5-32B-Instruct-AWQ,To evaluate the expression \(\left\lceil3\lef...,"{'nodes': [{'id': 'n1', 'text': 'Evaluate the ...",NaN,Yes,1778932026,1.906247
4,4,5,Sam is hired for a 20-day period. On days that...,Qwen/Qwen2.5-32B-Instruct-AWQ,"To solve the problem, we need to determine ho...","{'id': 'n1', 'text': 'Define variables for day...",NaN,Yes,1778932026,1.906247


In [ ]:
llm_graphs_df['is_correct'].value_counts()

,count
is_correct,
Yes,10132
No,2368


In [ ]:
llm_graphs_df['is_correct']

,is_correct
0,Yes
1,Yes
2,Yes
3,Yes
4,Yes
...,...
12495,Yes
12496,Yes
12497,Yes
12498,Yes


In [ ]:
from dataclasses import dataclass
from typing import Iterable, Optional
from sentence_transformers import SentenceTransformer
import numpy as np


@dataclass
class EmbeddingConfig:
    model_name: str = "sentence-transformers/all-mpnet-base-v2"
    batch_size: int = 32
    device: Optional[str] = None


class SentenceTransformerEmbedder:

    def __init__(self, cfg: EmbeddingConfig):
        self.cfg = cfg or EmbeddingConfig()
        self.model = SentenceTransformer(self.cfg.model_name, device=self.cfg.device)

    def encode(self, texts: Iterable[str]) -> np.ndarray:
        arr = self.model.encode(
            list(texts),
            batch_size=self.cfg.batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,

        )
        return np.asarray(arr)


def cosine_similarity_matrix(X: np.ndarray) -> np.ndarray:
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    return Xn @ Xn.T

In [ ]:
import networkx as nx

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
device = "cuda"

embedder = SentenceTransformerEmbedder(
        EmbeddingConfig(model_name=model_name, device=device)
    )

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

def load_graph_json(llm_graphs_df, graph_id, embedder):

    G = nx.DiGraph()
    row = llm_graphs_df[llm_graphs_df.id == graph_id]

    is_correct = True
    is_path = None

    if 'nodes' in row['graph'].values[0]:
      for idx, node in enumerate(row['graph'].values[0]['nodes']):
          if 'id' in node:
            embedding = embedder.encode([node['text']])
            embedding = embedding.reshape(embedding.shape[1])
            G.add_node(node['id'], embedding = embedding)
          else:
            is_correct = False

      if is_correct:
        for idx, edge in enumerate(row['graph'].values[0]['edges']):
            u, v = edge['source'], edge['target']
            if u in G.nodes and v in G.nodes():
              weight = cosine_distances([G.nodes[u]['embedding']], [G.nodes[v]['embedding']])[0][0]
              G.add_edge(u, v, weight=float(weight))

        is_path = is_path_graph(G)

    return {
        'graph': G,
        'num_nodes': len(G.nodes()),
        'num_edges': len(G.edges()),
        'is_path': is_path
    }

In [ ]:
import json
import pandas as pd
import scipy.sparse as sp
import pickle
from tqdm import tqdm
from pathlib import Path

def ensure_dir(path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

traces_path = '/content/all_traces_math_qwen32b_vllm_final.jsonl'
traces = []

with open(traces_path, 'r') as f:
    for line in f:
      if line.strip():
          trace = json.loads(line)
          traces.append(trace)

traces_df = pd.DataFrame(traces)

for id in tqdm(enumerate(traces_df['id'].tolist())):
  graph_data = load_graph_json(llm_graphs_df, id[1], embedder)
  if graph_data['num_nodes'] > 0 and graph_data['num_edges'] > 0 and graph_data['is_path'] == False:
    pid = str(id[1]).replace('/', '_')
    graph_path = Path('graph_dir')/ f'{pid}.pkl'
    G = graph_data['graph']
    ensure_dir(graph_path)
    with open(graph_path, 'wb') as f:
        pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)

12500it [20:23, 10.21it/s]


In [ ]:
llm_graphs_df

,Unnamed: 0,id,statement,model,trace,graph,reference_solution,is_correct,timestamp,processing_time
0,0,1,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",Qwen/Qwen2.5-32B-Instruct-AWQ,To ensure the piecewise function \( f(x) \) i...,"{'nodes': [{'id': 'n1', 'text': 'Check continu...",NaN,Yes,1778932026,1.906247
1,1,2,A rectangular band formation is a formation wi...,Qwen/Qwen2.5-32B-Instruct-AWQ,"To solve the problem, we need to find the lar...","{'nodes': [{'id': 'n1', 'text': 'Initial forma...",NaN,Yes,1778932026,1.906247
2,2,3,What is the degree of the polynomial $(4 +5x^3...,Qwen/Qwen2.5-32B-Instruct-AWQ,To determine the degree of the polynomial \(4...,"{'nodes': [{'id': 'n1', 'text': 'Combine const...",NaN,Yes,1778932026,1.906247
3,3,4,Evaluate $\left\lceil3\left(6-\frac12\right)\r...,Qwen/Qwen2.5-32B-Instruct-AWQ,To evaluate the expression \(\left\lceil3\lef...,"{'nodes': [{'id': 'n1', 'text': 'Evaluate the ...",NaN,Yes,1778932026,1.906247
4,4,5,Sam is hired for a 20-day period. On days that...,Qwen/Qwen2.5-32B-Instruct-AWQ,"To solve the problem, we need to determine ho...","{'id': 'n1', 'text': 'Define variables for day...",NaN,Yes,1778932026,1.906247
...,...,...,...,...,...,...,...,...,...,...
12495,76,6573,A $2\times 3$ rectangle and a $3\times 4$ rect...,Qwen/Qwen2.5-32B-Instruct-AWQ,To find the smallest possible area of the squ...,"{'nodes': [{'id': 'n1', 'text': 'Determine the...",NaN,Yes,1778972726,1.270443
12496,77,6574,"Amanda, Ben, and Carlos share a sum of money. ...",Qwen/Qwen2.5-32B-Instruct-AWQ,"To solve the problem, we will follow these st...","{'nodes': [{'id': 'n1', 'text': 'Identify the ...",NaN,Yes,1778972726,1.270443
12497,78,6575,A round-robin tennis tournament consists of ea...,Qwen/Qwen2.5-32B-Instruct-AWQ,To determine the number of matches in an 8-pe...,"{'nodes': [{'id': 'n1', 'text': 'Identify the ...",NaN,Yes,1778972726,1.270443
12498,79,6576,"Eugene, Brianna, and Katie are going on a run....",Qwen/Qwen2.5-32B-Instruct-AWQ,"To determine how fast Katie runs, we need to ...","{'nodes': [{'id': 'n1', 'text': 'Eugene runs a...",NaN,Yes,1778972726,1.270443


In [ ]:
!zip -r llm_graphs_pickle_math_Qwen2.5_32B.zip graph_dir

In [ ]:
!unzip /content/llm_graphs_pickle_math_Qwen2.5_32B.zip -d .

In [ ]:
!pip install torch_geometric
!pip install networkx

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool

class GNNBinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_weight, batch):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)
        graph_embedding = global_mean_pool(x, batch)
        out = self.classifier(graph_embedding)
        return out

In [ ]:
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data as PyGData
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from sklearn.metrics.pairwise import cosine_distances
import networkx as nx
import os
import json
import pandas as pd
import numpy as np
import pickle

class GraphBinaryDataset(Dataset):

    def __init__(self, dataset, graph_dir, traces_path, preload=False, transform=None):
        self.dataset = dataset
        self.graph_dir = graph_dir
        self.transform = transform
        self.preload = preload

        self.graph_files = [f for f in os.listdir(graph_dir)
                           if f.endswith(('.graphml')) or f.endswith(('.pkl')) ]

        print(self.graph_files)

        if os.path.exists(traces_path):
            traces = []
            with open(traces_path, 'r') as f:
               for line in f:
                  if line.strip():
                      trace = json.loads(line)
                      traces.append(trace)

            traces_df = pd.DataFrame(traces)
            traces_df['is_correct'] = traces_df.apply(lambda x: 'Yes' if (x.is_correct == 'Yes') else 'No', axis = 1)
            traces_df['is_correct'] = traces_df['is_correct'].apply(lambda x: 0.0 if x == 'Yes' else 1.0)
            self.traces = traces_df[['id', 'is_correct']]

        if preload:
            print("Preloading all graphs into memory...")
            self.data = []
            for idx in range(len(self.graph_files)):
                if idx % 100 == 0:
                    print(f"  Loading {idx}/{len(self.graph_files)}")
                pyg_data, label = self._load_item(idx)
                self.data.append((pyg_data, label))
            print(f"Preloaded {len(self.data)} graphs")


    def __len__(self):
        return len(self.graph_files)

    def _load_item(self, idx):
        graph_file = self.graph_files[idx]
        graph_path = os.path.join(self.graph_dir, graph_file)

        if self.dataset == 'math500':
            id = graph_file.split('.')[0]
            self.traces['id'] = self.traces['id'].apply(lambda x: x.replace('/', '_').split('.')[0])
            label = self.traces[self.traces.id == id]['is_correct'].values[0]
        else:
            id = graph_path.split('.')[0].split('/')[1]
            label = self.traces[self.traces.id == int(id)]['is_correct'].values[0]

        if graph_file.endswith('.graphml'):
          G = nx.read_gml(graph_path)
        if graph_file.endswith('.pkl'):
          with open(graph_path, 'rb') as f:
            G = pickle.load(f)

        pyg_data = self._nx_to_pyg(G)

        if isinstance(label, (int, float)):
            label = torch.tensor([label], dtype=torch.float32)

        return pyg_data, label

    def __getitem__(self, idx):
        if self.preload:
            pyg_data, label = self.data[idx]
            return pyg_data, label.clone() if isinstance(label, torch.Tensor) else label
        else:
            return self._load_item(idx)

    def _nx_to_pyg(self, G, edge_metric=None):

        nodes = list(G.nodes())

        node_to_idx = {node: i for i, node in enumerate(nodes)}


        node_features = []
        for node in nodes:
            if 'embedding' in G.nodes[node]:
                feat = np.array(G.nodes[node]['embedding'])

            node_features.append(feat)


        node_features = np.array(node_features)

        x = torch.tensor(node_features, dtype=torch.float)

        edges = []
        edge_weights = []

        for u, v in G.edges():

            edges.append([node_to_idx[u], node_to_idx[v]])
            if edge_metric:
              weight = cosine_distances([G.nodes[u]['embedding']], [G.nodes[v]['embedding']])[0][0]
            else:
              weight = G[u][v].get('weight', 1.0)

            edge_weights.append(weight)

            if not G.is_directed():
                edges.append([node_to_idx[v], node_to_idx[u]])
                edge_weights.append(weight)

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_weight = torch.tensor(edge_weights, dtype=torch.float) if edge_weights else None

        pyg_data = PyGData(x=x, edge_index=edge_index, edge_weight=edge_weight)

        return pyg_data


def main():
    GRAPH_DIR = 'graph_dir'
    traces_path = '/content/all_traces_math_qwen32b_vllm_final.jsonl'
    dataset = GraphBinaryDataset('math', GRAPH_DIR, traces_path)
    train_size = int(0.7 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size

    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size, test_size]
    )

    BATCH_SIZE = 32

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    return dataset, train_loader, val_loader, test_loader


if __name__ == "__main__":
    dataset, train_loader, val_loader, test_loader = main()

['6835.pkl', '10236.pkl', '8817.pkl', '5706.pkl', '7249.pkl', '9962.pkl', '5666.pkl', '4391.pkl', '11573.pkl', '9952.pkl', '12251.pkl', '10277.pkl', '317.pkl', '248.pkl', '1292.pkl', '6380.pkl', '10946.pkl', '4670.pkl', '6037.pkl', '9004.pkl', '1820.pkl', '6959.pkl', '3833.pkl', '9323.pkl', '4580.pkl', '9961.pkl', '3610.pkl', '3007.pkl', '8501.pkl', '8769.pkl', '12029.pkl', '1810.pkl', '4217.pkl', '2264.pkl', '10427.pkl', '1414.pkl', '1741.pkl', '3430.pkl', '1365.pkl', '3229.pkl', '12203.pkl', '7766.pkl', '9610.pkl', '3779.pkl', '718.pkl', '6179.pkl', '1876.pkl', '5239.pkl', '11480.pkl', '2354.pkl', '11555.pkl', '10450.pkl', '9042.pkl', '10084.pkl', '998.pkl', '10051.pkl', '5646.pkl', '6708.pkl', '9835.pkl', '9000.pkl', '10951.pkl', '5017.pkl', '5062.pkl', '2986.pkl', '6347.pkl', '125.pkl', '6553.pkl', '4869.pkl', '2300.pkl', '8776.pkl', '9329.pkl', '10043.pkl', '3688.pkl', '9321.pkl', '8003.pkl', '2682.pkl', '3732.pkl', '8885.pkl', '6917.pkl', '3859.pkl', '10922.pkl', '1650.pkl', '125

In [ ]:
len(dataset)

7149

In [ ]:
import numpy as np

def calculate_metrics(outputs, labels):

    probs = torch.sigmoid(outputs).detach().cpu().numpy().flatten()
    preds = (probs > 0.5).astype(int)

    labels_np = labels.detach().cpu().numpy().flatten()
    outputs = outputs.detach().cpu().numpy()

    print(np.unique(preds))

    metrics = {}

    metrics['accuracy'] = accuracy_score(labels_np, preds)

    metrics['recall'] = recall_score(labels_np, preds, zero_division=0)

    metrics['roc_auc'] = roc_auc_score(labels_np, outputs)

    metrics['pr_auc'] = average_precision_score(labels_np, outputs)

    return metrics

In [ ]:
def evaluate(model, data_loader, criterion, device='cpu'):

    model.eval()
    total_loss = 0
    all_outputs = []
    all_labels = []

    with torch.no_grad():
        for graphs, labels in data_loader:

            graphs = graphs.to(device)
            labels = labels.to(device)

            outputs = model(graphs.x, graphs.edge_index, graphs.edge_weight, graphs.batch)
            #outputs = model(graphs.x, graphs.edge_index, graphs.batch)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            all_outputs.append(outputs)
            all_labels.append(labels)


    all_outputs = torch.cat(all_outputs, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    metrics = calculate_metrics(all_outputs, all_labels)
    avg_loss = total_loss / len(data_loader)

    return avg_loss, metrics

In [ ]:
!pip install early-stopping-pytorch

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, average_precision_score
from early_stopping_pytorch import EarlyStopping

early_stopping = EarlyStopping(patience=7, verbose=True)

model = GNNBinaryClassifier(input_dim=768, hidden_dim=64)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

losses = []

num_epochs = 10

val_losses = []


iter = 0
while True:

  all_train_outputs = []
  all_train_labels = []
  losses = []

  for batch_idx, (graphs, labels) in enumerate(train_loader):
      outputs = model(graphs.x, graphs.edge_index, graphs.edge_weight, graphs.batch)
      loss = criterion(outputs, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      all_train_outputs.append(outputs)
      all_train_labels.append(labels)

      losses.append(loss.item())

  all_train_outputs = torch.cat(all_train_outputs, dim=0)
  all_train_labels = torch.cat(all_train_labels, dim=0)

  mean_loss = np.mean(np.array(losses))
  train_metrics = calculate_metrics(all_train_outputs, all_train_labels)

  print(f"Epoch {iter+1}/{num_epochs}")
  print(f"{'='*60}")
  print(f"Train Loss: {mean_loss:.4f}")
  print(f"\nTrain Metrics:")
  print(f"  Accuracy: {train_metrics['accuracy']:.4f}")
  print(f"  Recall:   {train_metrics['recall']:.4f}")
  print(f"  ROC-AUC:  {train_metrics['roc_auc']:.4f}")
  print(f"  PR-AUC:   {train_metrics['pr_auc']:.4f}")

  val_loss, val_metrics = evaluate(model, val_loader, criterion)
  val_losses.append(val_loss)

  print(f"\nVal Metrics:")
  print(f"  Accuracy: {val_metrics['accuracy']:.4f}")
  print(f"  Recall:   {val_metrics['recall']:.4f}")
  print(f"  ROC-AUC:  {val_metrics['roc_auc']:.4f}")
  print(f"  PR-AUC:   {val_metrics['pr_auc']:.4f}")
  print(f"{'='*60}\n")

  iter += 1

  early_stopping(val_loss, model)
  if early_stopping.early_stop:
      print("Early stopping triggered")
      break

[0]
Epoch 1/10
Train Loss: 0.4651

Train Metrics:
  Accuracy: 0.8147
  Recall:   0.0000
  ROC-AUC:  0.6420
  PR-AUC:   0.2520
[0]

Val Metrics:
  Accuracy: 0.7969
  Recall:   0.0000
  ROC-AUC:  0.7321
  PR-AUC:   0.3923

Validation loss decreased (inf --> 0.451701).  Saving model ...
[0 1]
Epoch 2/10
Train Loss: 0.4291

Train Metrics:
  Accuracy: 0.8145
  Recall:   0.0162
  ROC-AUC:  0.7315
  PR-AUC:   0.3773
[0 1]

Val Metrics:
  Accuracy: 0.7969
  Recall:   0.0345
  ROC-AUC:  0.7398
  PR-AUC:   0.4182

Validation loss decreased (0.451701 --> 0.444879).  Saving model ...
[0 1]
Epoch 3/10
Train Loss: 0.4130

Train Metrics:
  Accuracy: 0.8175
  Recall:   0.0809
  ROC-AUC:  0.7557
  PR-AUC:   0.4219
[0 1]

Val Metrics:
  Accuracy: 0.8081
  Recall:   0.1655
  ROC-AUC:  0.7525
  PR-AUC:   0.4502

Validation loss decreased (0.444879 --> 0.434857).  Saving model ...
[0 1]
Epoch 4/10
Train Loss: 0.4021

Train Metrics:
  Accuracy: 0.8265
  Recall:   0.1510
  ROC-AUC:  0.7737
  PR-AUC:   0.4566

In [ ]:
test_loss, test_metrics = evaluate(model, test_loader, criterion)
print("TEST SET RESULTS:")
for metric, value in test_metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

[0 1]
TEST SET RESULTS:
  ACCURACY: 0.7980
  RECALL: 0.1661
  ROC_AUC: 0.7180
  PR_AUC: 0.3734


In [ ]:
torch.save(model.state_dict(), 'best_math_qwen2.5_32B_attention_graphs_model_32_bs_weights.pth')

torch.save(model, 'best_math_qwen2.5_32B_attention_graphs_full_model_32_bs.pth')